# Konvertera bilder till koordinate med mediapipe
Jatg började mned att köra dettta i asl_training. Men det blev ltie tungt, så det fick en egen notbook som kör det

<img src="https://ai.google.dev/static/mediapipe/images/solutions/hand-landmarks.png" width="300" alt="beskrivning">
Dataset från :https://www.kaggle.com/datasets/debashishsau/aslamerican-sign-language-aplhabet-dataset

In [11]:
"""https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker"""
""" DARTASET är från """
import os
import csv
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from tqdm import tqdm

#globaler
DATA_MAPP  = '/media/macke/D/school/NBIHAK-PAIHT25D/examensuppgift/asl_alphabet_train'#'\\media\\macke\\D\\school\\NBIHAK-PAIHT25D\\examensuppgift\\asl_alphabet_train' #fick inte riktigt sökbvägen att fungera så kör med absolut just nu..
###W0000 00:00:1778749039.714125   15935 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors. undersök#
CSV_FIL = 'asl_landmarks_700.csv'
MAX_BILDER = 700  #1000 # antal bilder per bokstav
RESYNK_TRESHOLD = 0.75


# lite trixigt att hitta koordinater

In [12]:

## ladda in mediapipe ochj lägg på hand_landmarker.task.###
base_options = mp_python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)

##CSV headers, vi hämtar 21 parametrar/landmarks för varje "position" så vi skapar en header med alla 21 punkter x1..x21
header =  []
for i in range(21):
    header.append(f'x{i}')
    header.append(f'y{i}')

#lable
header.append('label')

# Loopa igenom alla bokstavsmappar
#W0000 00:00:1778749039.714125   15935 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors. undersök#
rader        = []
missade      = 0
totalt       = 0
bokstavsmapp = sorted(os.listdir(DATA_MAPP))

I0000 00:00:1778760173.456143  173410 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1778760173.466648  173427 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: llvmpipe (LLVM 20.1.2, 256 bits)
W0000 00:00:1778760173.482189  173412 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778760173.492258  173413 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## loopa igneom alla mappar
scriptet hämtar bara 10% av alla bilder som finns för barje bokstav förf att inte göra modellen för stor.
Dock, jag fick inte utslag på vissa av dessa (N & M Looking at you), så extrahera med enn fallback som hämtar större set om det är under tex 75% träffar.

In [13]:
print(f'Hittade {len(bokstavsmapp)} mappar')


"""
for bokstav in bokstavsmapp:
    mapp_path = os.path.join(DATA_MAPP, bokstav)
    #kraschade några gånger på .DS etc, verifiera att det är en mapp
    if not os.path.isdir(mapp_path):
        continue
    #h'ämta lista bilderna för varje mapp,
    bilder = [f for f in os.listdir(mapp_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:MAX_BILDER]

    for filnamn in tqdm(bilder, desc=f'{bokstav} ({len(bilder)} bilder)'):
        file_path = os.path.join(mapp_path, filnamn)

        try:
            #här kör konvertering
            mp_image = mp.Image.create_from_file(file_path) #läser in bilden
            resultat = detector.detect(mp_image) #här detekterar den en hand med hjälpa av hand_landmarker.task och får ut "landmarks" som är typ koordinater.
        except Exception:
            missade += 1
            continue

        if not resultat.hand_landmarks:
            missade += 1
            continue
        
        landmarks = resultat.hand_landmarks[0]
        rad = []
        for lm in landmarks:
            rad.append(round(lm.x, 6))
            rad.append(round(lm.y, 6))
        rad.append(bokstav)

        rader.append(rad)
        totalt += 1
"""

def extrahera_landmarks(bilder, mapp_path):
    resultat_rader = []
    for filnamn in bilder:
        file_path = os.path.join(mapp_path, filnamn)
        try:
            mp_image = mp.Image.create_from_file(file_path)
            resultat = detector.detect(mp_image)
        except Exception:
            continue
        if not resultat.hand_landmarks:
            continue
        landmarks = resultat.hand_landmarks[0]
        rad = []
        for lm in landmarks:
            rad.append(round(lm.x, 6))
            rad.append(round(lm.y, 6))
        rad.append(bokstav)
        resultat_rader.append(rad)
    return resultat_rader

def fetch_images(mapp_path, bokstav, max_bilder=MAX_BILDER, treshold=RESYNK_TRESHOLD):
    bilder = [f for f in os.listdir(mapp_path) 
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:max_bilder]
    resultat = extrahera_landmarks(bilder, mapp_path)
    print(f"{bokstav} hittade: {len(resultat)}")
    if len(resultat) < max_bilder * treshold:
        print(f'{bokstav}: under {treshold*100}% träff — kör om med 1.5xMAX_BILDER bilder')
        bilder = [f for f in os.listdir(mapp_path) 
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:int(MAX_BILDER*1.5)] ##vill ha int, kom ihåg
        resultat = extrahera_landmarks(bilder, mapp_path)
        print(f"runda 2 hittade: {len(resultat)}")
    return resultat


for bokstav in tqdm(bokstavsmapp, desc='Bokstäver'):
    mapp_path = os.path.join(DATA_MAPP, bokstav)
    if not os.path.isdir(mapp_path):
        continue
    rader.extend(fetch_images(mapp_path, bokstav))


Hittade 29 mappar


Bokstäver:   0%|          | 0/29 [00:00<?, ?it/s]

A hittade: 424
A: under 75.0% träff — kör om med 1.5xMAX_BILDER bilder


Bokstäver:   3%|▎         | 1/29 [00:22<10:43, 23.00s/it]

runda 2 hittade: 631
B hittade: 496
B: under 75.0% träff — kör om med 1.5xMAX_BILDER bilder


...

runda 2 hittade: 573
nothing hittade: 0
nothing: under 75.0% träff — kör om med 1.5xMAX_BILDER bilder


Bokstäver:  97%|█████████▋| 28/29 [09:06<00:19, 19.47s/it]

runda 2 hittade: 0
space hittade: 314
space: under 75.0% träff — kör om med 1.5xMAX_BILDER bilder


Bokstäver: 100%|██████████| 29/29 [09:27<00:00, 19.57s/it]

runda 2 hittade: 470


iteration 5..mijoner?

spara fil

In [14]:
# file
with open(CSV_FIL, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(rader)

detector.close()

print(f'rader:{len(rader)} fil:{CSV_FIL}')
print(f'misslyckade {missade}')

rader:18152 fil:asl_landmarks_700.csv
misslyckade 0
